# Day4: Differential Expression

このノートブックでは、差次的発現解析で出てくる基本指標を読み解きます。

このトピックが役立つ場面:
薬剤処理や刺激の前後で、どの遺伝子が本当に変化したのかを候補として絞り込みたい場面です。

このトピックで解く課題:
treated によって本当に変化した遺伝子はどれかを、fold change と有意性の両方を見ながら判断します。

## 差次的発現解析とは

差次的発現解析は、control と treated の間で、どの遺伝子の発現がどの程度変わったかを統計的に評価する工程です。

ここでは本格的な統計モデルではなく、結果の表をどう読むかに焦点を当てます。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

deg = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "log2FoldChange": [1.2, 1.1, 0.1, 0.0, 2.8],
    "pvalue": [0.04, 0.03, 0.80, 0.95, 0.0005],
    "padj": [0.08, 0.06, 0.90, 0.95, 0.002]
}).set_index("gene")

deg

## 重要な列

- `log2FoldChange`: 発現変化の大きさと方向
- `pvalue`: その差が偶然だけで起きたと考えたときの確率
- `padj`: 多重検定補正後の値

`log2FoldChange` が正なら treated で高く、負なら treated で低いことを示します。

In [ ]:
deg["minus_log10_padj"] = -np.log10(deg["padj"])
deg.sort_values(["padj", "log2FoldChange"])

## Volcano plot

Volcano plot は、横軸に発現変化の大きさ、縦軸に統計的な強さを置いた図です。右上にある遺伝子ほど「大きく上昇し、しかも有意」であることを示します。

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(deg["log2FoldChange"], deg["minus_log10_padj"], color="#4C78A8", s=70)
for gene, row in deg.iterrows():
    plt.text(row["log2FoldChange"] + 0.03, row["minus_log10_padj"] + 0.03, gene)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("log2 fold change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano plot")
plt.tight_layout()
plt.show()

この例では `IL6` が右上に位置し、treated で強く上昇していて、統計的にも目立つ遺伝子だと読めます。`GAPDH` や `ACTB` は大きな変化がなく、比較的安定です。

## ここまでの要点

- 差次的発現解析では「どれだけ変わったか」と「どれくらい信頼できるか」を同時に見る
- `log2FoldChange` は変化量、`padj` は有意性の目安
- Volcano plot はその二つを一枚で見せる
- 次は遺伝子リストを生物学的意味へつなげる